# FietsRoute: Dijkstra nas ruas de Amsterdã

**Equipe:** Ana Alice, Caio Vasconcelos e Yasmin Barata

O Dijkstra, a fila de prioridade e a BFS são código da equipe. O OSMnx só baixa as ruas, e o folium e o matplotlib só desenham o resultado.

*Dados do mapa © colaboradores do OpenStreetMap (licença ODbL).*

## Preparação

In [ ]:
import time

from IPython.display import Image, display

from mapa.consultas import ponto_mais_proximo, trechos_por_nome
from mapa.desenhar import desenhar_pontos, desenhar_rota
from mapa.gerar_gifs import FATOR_LOTADO, VELOCIDADE_M_POR_MIN, aplicar_lotacao, carregar_demo, trechos_dos_empecilhos
from nucleo.dijkstra import bfs, caminho, comprimento_rota, custo_rota, dijkstra, dijkstra_linear
from nucleo.grafo import carregar_json
from testes.referencia import dijkstra_networkx


def metros(valor):
    return "∞" if valor == float("inf") else f"{valor:,.0f}".replace(",", ".")


def mostrar_passos(passos, destino):
    for numero, passo in enumerate(passos, 1):
        print(f"{numero}. Fixa {passo['fixado']} ({metros(passo['custo'])})")
        if passo["fixado"] == destino:
            print("   destino fixado: para aqui")
        for vizinho, antes, proposto, melhorou in passo["tentativas"]:
            if melhorou:
                print(f"   {vizinho}: {metros(antes)} → {metros(proposto)}")
            else:
                print(f"   {vizinho} por {passo['fixado']} daria {metros(proposto)}, não melhora {metros(antes)}")


def calcular(grafo, origem, destino):
    _, anterior, passos = dijkstra(grafo, origem, destino)
    return caminho(anterior, destino), passos


def mostrar_rota(grafo, rota, titulo, empecilho=False):
    distancia = comprimento_rota(grafo, rota)
    minutos = f"{distancia / VELOCIDADE_M_POR_MIN:.1f}".replace(".", ",")
    print(titulo)
    print(f"  Distância real: {metros(distancia)} m")
    if empecilho:
        print(f"  Custo: {metros(custo_rota(grafo, rota))} metros equivalentes")
        print(f"  Tempo estimado: {minutos} min a 15 km/h, sem considerar a lotação")
    else:
        print(f"  Tempo estimado: {minutos} min a 15 km/h")

## 1. Exemplo dos slides

Passo a passo de Estação Central até Parque Vondel.

In [ ]:
exemplo = carregar_json("dados/exemplo_slides.json")
rota_exemplo, passos_exemplo = calcular(exemplo, "Estação Central", "Parque Vondel")
mostrar_passos(passos_exemplo, "Parque Vondel")
print("\nRota:", " → ".join(rota_exemplo), f"= {metros(custo_rota(exemplo, rota_exemplo))} m")

In [ ]:
menos_trechos = caminho(bfs(exemplo, "Estação Central", "Parque Vondel"), "Parque Vondel")
print("Menos trechos (BFS):", " → ".join(menos_trechos))
print(f"  {len(menos_trechos) - 1} trechos, {metros(custo_rota(exemplo, menos_trechos))} m")
print("Menor custo (Dijkstra):", " → ".join(rota_exemplo))
print(f"  {len(rota_exemplo) - 1} trechos, {metros(custo_rota(exemplo, rota_exemplo))} m")

In [ ]:
ponte_fechada = exemplo.copiar()
ponte_fechada.remover_trecho("Praça Dam", "Praça Leidse")
nova_exemplo, _ = calcular(ponte_fechada, "Estação Central", "Parque Vondel")
print("Ponte Praça Dam – Praça Leidse fechada")
print("Nova rota:", " → ".join(nova_exemplo))
print(f"Custo: {metros(custo_rota(exemplo, rota_exemplo))} → {metros(custo_rota(ponte_fechada, nova_exemplo))} m")

In [ ]:
FATOR_EVENTO = 3

evento = ponte_fechada.copiar()
evento.alterar_custo("Praça Leidse", "Parque Vondel", FATOR_EVENTO)
rota_evento, _ = calcular(evento, "Estação Central", "Parque Vondel")
print(f"Evento na entrada do parque: Praça Leidse – Parque Vondel × {FATOR_EVENTO}")
print(f"Custo do trecho: {metros(ponte_fechada.trecho('Praça Leidse', 'Parque Vondel')['custo'])} → {metros(evento.trecho('Praça Leidse', 'Parque Vondel')['custo'])} m")
print("Nova rota:", " → ".join(rota_evento), f"= {metros(custo_rota(evento, rota_evento))} m")
print(f"A rota anterior agora custaria {metros(custo_rota(evento, nova_exemplo))} m")

In [ ]:
try:
    exemplo.alterar_custo("Praça Leidse", "Parque Vondel", -3)
except ValueError as erro:
    print("ValueError:", erro)

## 2. Mapa real

In [ ]:
mapa_real, pontos = carregar_demo()
empecilhos = trechos_dos_empecilhos(mapa_real, pontos)
pontos_por_nome = {ponto["nome"]: ponto for ponto in pontos.values()}


def cruzamento(nome):
    if nome not in pontos_por_nome:
        raise ValueError(f"Ponto desconhecido: {nome}. Opções: {', '.join(pontos_por_nome)}")
    ponto = pontos_por_nome[nome]
    return ponto_mais_proximo(mapa_real, ponto["lat"], ponto["lon"])


print(f"{metros(len(mapa_real.pontos()))} cruzamentos e {metros(len(mapa_real.trechos()))} trechos")

## 3. Origem e destino

In [ ]:
ORIGEM = "Estação Central"
DESTINO = "Museu Van Gogh"

origem, destino = cruzamento(ORIGEM), cruzamento(DESTINO)
desenhar_pontos(pontos.values(), pontos_por_nome[ORIGEM], pontos_por_nome[DESTINO])

## 4. Rota de menor custo

In [ ]:
inicio = time.perf_counter()
rota, passos = calcular(mapa_real, origem, destino)
milissegundos = (time.perf_counter() - inicio) * 1000
mostrar_rota(mapa_real, rota, f"{ORIGEM} → {DESTINO}")
print(f"  {metros(len(passos))} cruzamentos fixados em {milissegundos:.0f} ms")
desenhar_rota(mapa_real, rota, pontos.values())

In [ ]:
display(Image(filename="gifs/rota.gif"))

## 5. Empecilho 1: passagem do Museu Nacional lotada

In [ ]:
def recalcular(nome, trechos, fator=FATOR_LOTADO):
    cenario = aplicar_lotacao(mapa_real, trechos, fator)
    nova, _ = calcular(cenario, origem, destino)
    mostrar_rota(cenario, nova, f"{nome} (custo × {fator})", empecilho=True)
    print(f"  A rota anterior agora custaria {metros(custo_rota(cenario, rota))} metros equivalentes")
    if nova == rota:
        print("  A rota não mudou: ela continua sendo a de menor custo")
    return desenhar_rota(mapa_real, nova, pontos.values(), rota, trechos)


recalcular(*empecilhos["empecilho1"])

In [ ]:
display(Image(filename="gifs/empecilho1.gif"))

## 6. Empecilho 2: Praça Dam lotada por um evento

In [ ]:
recalcular(*empecilhos["empecilho2"])

In [ ]:
display(Image(filename="gifs/empecilho2.gif"))

## 7. Alterar o custo de qualquer rua

Troque `RUA` e `FATOR`. Um fator maior que 1 encarece a rua, um fator entre 0 e 1 barateia, e um fator negativo é recusado.

In [ ]:
RUA = "Weteringschans"
FATOR = 3

try:
    trechos_rua = trechos_por_nome(mapa_real, RUA)
    if not trechos_rua:
        raise ValueError(f"Nenhuma rua encontrada com o nome: {RUA}")
    mapa_custo = recalcular(RUA, trechos_rua, FATOR)
except ValueError as erro:
    print("ValueError:", erro)
    mapa_custo = None
mapa_custo

## 8. Qualquer origem e destino

In [ ]:
NOVA_ORIGEM = "Parque Vondel"
NOVO_DESTINO = "Magere Brug"

nova_rota, _ = calcular(mapa_real, cruzamento(NOVA_ORIGEM), cruzamento(NOVO_DESTINO))
mostrar_rota(mapa_real, nova_rota, f"{NOVA_ORIGEM} → {NOVO_DESTINO}")
desenhar_rota(mapa_real, nova_rota, pontos.values())

## 9. Extra: comparações no mapa real

In [ ]:
rota_bfs = caminho(bfs(mapa_real, origem, destino), destino)
print(f"Dijkstra: {len(rota) - 1} trechos, {metros(comprimento_rota(mapa_real, rota))} m")
print(f"BFS: {len(rota_bfs) - 1} trechos, {metros(comprimento_rota(mapa_real, rota_bfs))} m")

inicio = time.perf_counter()
dijkstra(mapa_real, origem, destino)
com_heap = (time.perf_counter() - inicio) * 1000
inicio = time.perf_counter()
dijkstra_linear(mapa_real, origem, destino)
sem_heap = (time.perf_counter() - inicio) * 1000
com_heap_texto = f"{com_heap:.1f}".replace(".", ",")
print(f"\nCom heap: {com_heap_texto} ms | Sem heap: {sem_heap:.0f} ms | cerca de {sem_heap / com_heap:.0f} vezes mais lento")

custo_biblioteca, _ = dijkstra_networkx(mapa_real, origem, destino)
igual = abs(custo_rota(mapa_real, rota) - custo_biblioteca) < 1e-6
print(f"\nValidação com NetworkX: {metros(custo_biblioteca)} m, {'mesmo custo' if igual else 'CUSTO DIFERENTE'} da rota da equipe")